# Fhl2-OE cardiomyocyte snRNA-seq 

Corrected copy of `scanpy_CM_Fhl2.ipynb`. The analysis steps and figure formats are
unchanged; three defects are fixed.

| # | Original | Corrected here |
|---|---|---|
| 1 | `adata.concatenate([adata, adata_fhl2])` put `adata` in **twice** (Ctrl/Ex/SED counted 2×, Fhl2OE 1×) | `adata.concatenate(adata_fhl2)` |
| 2 | violin stars from `custom_pvalues = [1e-10]*n` | Kruskal–Wallis + pairwise Mann–Whitney U computed on the data |
| 3 | leiden IDs `["16","36","43"]` dropped by hard-coded number; no seeds | seeds fixed; exclusion is the reviewable `EXCLUDE_CLUSTERS` parameter |

Nothing is written to the original `adata/` or `figures/` directories.

In [1]:
import os, json, warnings
import numpy as np, pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy.stats import kruskal, mannwhitneyu
from statannotations.Annotator import Annotator

SEED = 0
np.random.seed(SEED)

BASE     = "/Users/takahiro/Desktop/project/Reha/snRNAseq_scanpy"
FHL2_DIR = f"{BASE}/Fhl2OE"
OUT_AD   = f"{FHL2_DIR}/adata_fixed"      # new — originals untouched
OUT_FIG  = f"{FHL2_DIR}/figures_fixed"
os.makedirs(OUT_AD, exist_ok=True); os.makedirs(OUT_FIG, exist_ok=True)

sc.settings.verbosity = 1
sc.settings.figdir = OUT_FIG
sc.settings.set_figure_params(figsize=(6, 6), dpi=100)
plt.rcParams['axes.grid'] = False
warnings.filterwarnings("ignore")
print("scanpy", sc.__version__)

scanpy 1.10.3


## 1. Load inputs and concatenate (fix 1)

In [2]:
adata      = sc.read_h5ad(f"{BASE}/adata/CM_Ctrl_6W_analysed_raw.h5ad")
adata_fhl2 = sc.read_h5ad(f"{FHL2_DIR}/adata/CM_Fhl2OE.h5ad")

print("adata      :", adata.n_obs, dict(adata.obs['sample'].value_counts()))
print("adata_fhl2 :", adata_fhl2.n_obs, dict(adata_fhl2.obs['sample'].value_counts()))
n_expected = adata.n_obs + adata_fhl2.n_obs

adata      : 10865 {'SED6W2': 2293, 'Ctrl2': 2000, 'Reha6W2': 1964, 'Ctrl1': 1935, 'Reha6W1': 1933, 'SED6W1': 740}
adata_fhl2 : 4891 {'Fhl2OE2': 2508, 'Fhl2OE1': 2383}


In [3]:
# FIX 1 — original was: adata.concatenate([adata, adata_fhl2])  -> adata entered twice.
# `.concatenate()` is a method: self is the first object, arguments are the rest.
adata = adata.concatenate(adata_fhl2).copy()

assert adata.n_obs == n_expected, f"{adata.n_obs} != {n_expected}"
print("concatenated:", adata.n_obs, "(expected", n_expected, ")")
print("per batch   :", dict(adata.obs['batch'].value_counts()))
print("per sample  :", dict(adata.obs['sample'].value_counts()))

concatenated: 15756 (expected 15756 )
per batch   : {'0': 10865, '1': 4891}
per sample  : {'Fhl2OE2': 2508, 'Fhl2OE1': 2383, 'SED6W2': 2293, 'Ctrl2': 2000, 'Reha6W2': 1964, 'Ctrl1': 1935, 'Reha6W1': 1933, 'SED6W1': 740}


## 2. QC overview (unchanged)

In [4]:
sc.pl.highest_expr_genes(adata, n_top=20, show=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
sc.pl.violin(adata, ['n_genes_by_counts'], groupby='sample', ax=axes[0], show=False, rotation=90)
sc.pl.violin(adata, ['total_counts'],      groupby='sample', ax=axes[1], show=False, rotation=90)
sc.pl.violin(adata, ['pct_counts_mt'],     groupby='sample', ax=axes[2], show=False, rotation=90)
plt.tight_layout(); plt.show()

sc.pl.violin(adata, ['Fhl2'], groupby='sample', rotation=90)
sc.pl.scatter(adata, x="total_counts", y="pct_counts_mt")

## 3. Normalisation, HVG, regression, PCA (unchanged)

In [5]:
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata.copy()

sc.pp.highly_variable_genes(adata, n_top_genes=2000)
sc.pl.highly_variable_genes(adata)

sc.pp.regress_out(adata, ["total_counts", "pct_counts_mt"])
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver="arpack", random_state=SEED)
sc.pl.pca_variance_ratio(adata, log=True)
adata.write_h5ad(f"{OUT_AD}/CM_Fhl2_beforecuration.h5ad")

## 4. Drop Ctrl, order conditions (unchanged)

In [6]:
adata = adata[adata.obs["type"] != "Ctrl"].copy()
adata.obs["type"] = pd.Categorical(adata.obs["type"].astype(str),
                                   categories=["SED", "Ex", "Fhl2OE"], ordered=True)
adata.obs["type"] = adata.obs["type"].cat.rename_categories(
    {"Fhl2OE": r"$\it{Fhl2}$ OE"})
print(adata.obs["type"].value_counts())

type
$\it{Fhl2}$ OE    4891
Ex                3897
SED               3033
Name: count, dtype: int64


## 5. Neighbours / UMAP / clustering (seeds fixed)

In [7]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=15, random_state=SEED)
sc.tl.umap(adata, spread=0.5, min_dist=0.5, random_state=SEED)
sc.pl.umap(adata, color=["type"], vmax=5)
sc.tl.leiden(adata, resolution=0.5, random_state=SEED)   # flavor = scanpy default (leidenalg), as in the original
sc.pl.umap(adata, color=["leiden"])
print("clusters:", adata.obs['leiden'].nunique())

clusters: 9


## 6. Cluster QC 

The original dropped leiden `["16","36","43"]` (labelled *low quality*). Those IDs belonged to
the duplicated run and mean nothing here, so the same clusters are re-identified from the data.
`EXCLUDE_CLUSTERS = "AUTO"` flags clusters that are outliers on detected genes / mitochondrial
content / cardiomyocyte-marker expression. **Check the table and the dotplot below, then set the
list explicitly if you disagree.**

In [8]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
sc.pl.violin(adata, ['n_genes_by_counts'], groupby='leiden', ax=axes[0], show=False)
sc.pl.violin(adata, ['total_counts'],      groupby='leiden', ax=axes[1], show=False)
sc.pl.violin(adata, ['pct_counts_mt'],     groupby='leiden', ax=axes[2], show=False, rotation=90)
plt.tight_layout(); plt.show()

sc.pl.dotplot(adata, [
  "Ttn", "Mybpc3",                                   # CM
  "Dcn", "Col1a1", "Postn",                          # FB
  "Frmd3", "Dlc1", "Myh11",                          # SMC
  "Pecam1", "Cdh5", "Vwf", "Npr3",                   # EC / endocardial
  "Ptprc", "Mrc1", "Cd163",                          # macrophage
  "Cd3e", "Skap1", "Cd79a", "Cd79b", "Il7r", "Kit",  # T / B
  "Lmnb1", "Slpi", "S100a9",                         # granulocyte
  "Nrxn1", "Nrxn3", "Upk3b", "Msln", "Gpc3",
  "Plin1", "Xkr4", "Acta2", "Ms4a1", "Ncr1", "Vtn",
  "Colec11", "Steap4", "Kcnj8", "Mmrn1", "Flt4"
], groupby="leiden", vmax=4)

In [9]:
EXCLUDE_CLUSTERS = "AUTO"   # or e.g. ["7", "12"]

def cluster_qc(ad):
    raw = ad.raw.to_adata()
    cm = np.asarray(raw[:, ["Ttn", "Mybpc3"]].X.todense()).mean(axis=1)
    df = pd.DataFrame({"leiden": ad.obs["leiden"].astype(str).values,
                       "n_genes": ad.obs["n_genes_by_counts"].values,
                       "pct_mt": ad.obs["pct_counts_mt"].values,
                       "cm_marker": cm})
    t = df.groupby("leiden").agg(n_cells=("n_genes", "size"),
                                 med_n_genes=("n_genes", "median"),
                                 med_pct_mt=("pct_mt", "median"),
                                 mean_cm=("cm_marker", "mean"))
    return t.sort_values("med_n_genes")

qc = cluster_qc(adata)
g_cut, mt_cut, cm_cut = qc.med_n_genes.median()*0.5, qc.med_pct_mt.median()*3, qc.mean_cm.median()*0.5
qc["flag"] = np.where((qc.med_n_genes < g_cut) | (qc.med_pct_mt > mt_cut) | (qc.mean_cm < cm_cut),
                      "LOW-QUALITY?", "")
print(f"thresholds: med_n_genes < {g_cut:.0f} | med_pct_mt > {mt_cut:.2f} | mean_cm < {cm_cut:.3f}")
print(qc.to_string())
auto = qc.index[qc["flag"] != ""].tolist()
drop = auto if EXCLUDE_CLUSTERS == "AUTO" else list(EXCLUDE_CLUSTERS)
print("\nexcluding:", drop or "(none)")

thresholds: med_n_genes < 874 | med_pct_mt > 0.00 | mean_cm < 1.787
        n_cells  med_n_genes  med_pct_mt   mean_cm          flag
leiden                                                          
7           126        767.5    0.228921  3.405473  LOW-QUALITY?
6           761        852.0    0.000000  3.573558  LOW-QUALITY?
2          1854       1623.0    0.000000  3.899729              
3          1795       1737.0    0.000000  4.054801              
8            12       1747.0    0.000000  3.382118              
4          1768       1899.0    0.000000  3.946100              
5           780       1981.0    0.000000  4.037968              
1          2317       2160.0    0.000000  3.358332              
0          2408       2348.0    0.000000  3.427312              

excluding: ['7', '6']


In [10]:
if drop:
    adata = adata[~adata.obs["leiden"].isin(drop)].copy()
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=15, random_state=SEED)
sc.tl.leiden(adata, resolution=0.2, random_state=SEED)   # flavor = scanpy default (leidenalg), as in the original
print("cells after exclusion:", adata.n_obs)
print(adata.obs["type"].value_counts())

cells after exclusion: 10934
type
$\it{Fhl2}$ OE    4706
Ex                3456
SED               2772
Name: count, dtype: int64


### Cell numbers for the Fig. 4e legend
The published legend reads SED 3,033 / Ex 3,897 / Fhl2 OE 2,329. Those SED/Ex values are the
pre-QC input counts and the Fhl2 OE value is half of what the duplicated run actually held.
Use the numbers printed above.

In [11]:
counts = adata.obs["type"].value_counts()
pd.DataFrame({"corrected_run": counts,
              "published_legend": pd.Series({"SED": 3033, "Ex": 3897, r"$\it{Fhl2}$ OE": 2329})})

,corrected_run,published_legend
$\it{Fhl2}$ OE,4706,2329
Ex,3456,3897
SED,2772,3033


## 7. Figure 4e — UMAP (same format)

In [12]:
colors = ["#2ca02c", "#ff7f0e", "#d62728"]
sc.settings.set_figure_params(figsize=(6, 6), dpi=100)
sc.pl.umap(adata, color=["type"], palette=colors, save="_CM_type_fixed.pdf")
sc.pl.umap(adata, color=["leiden"], save="_CM_leiden_fixed.pdf")

## 8. Gene scores (same five GO sets)

In [13]:
def load_set(name):
    with open(f"{FHL2_DIR}/geneset/{name}", "r") as f:
        d = json.load(f)
    return d[list(d.keys())[0]]["geneSymbols"]

cardiac_contraction = load_set("GOBP_CARDIAC_MUSCLE_CONTRACTION.v2025.1.Mm.json")
calcium_contraction = load_set("GOBP_REGULATION_OF_CARDIAC_MUSCLE_CONTRACTION_BY_CALCIUM_ION_SIGNALING.v2025.1.Mm.json")
metabolites         = load_set("GOBP_GENERATION_OF_PRECURSOR_METABOLITES_AND_ENERGY.v2025.1.Mm.json")
respiration         = load_set("GOBP_REGULATION_OF_AEROBIC_RESPIRATION.v2025.1.Mm.json")
acute_inflammation  = load_set("GOBP_ACUTE_INFLAMMATORY_RESPONSE.v2025.1.Mm.json")

# The original called score_genes twice (use_raw=False, then the default). The second call
# overwrote the first, so the published scores are the .raw-based ones — kept here.
for genes_, name in [(cardiac_contraction, "cardiac_contraction_score"),
                     (calcium_contraction, "calcium_signaling_score"),
                     (metabolites,         "metabolites_energy_score"),
                     (respiration,         "aerobic_respiration_score"),
                     (acute_inflammation,  "inflammation_score")]:
    sc.tl.score_genes(adata, genes_, score_name=name, random_state=SEED)

sc.pl.umap(adata, color=["cardiac_contraction_score", "metabolites_energy_score",
                         "aerobic_respiration_score"], save="_scores_fixed.pdf")
sc.pl.umap(adata, color=["inflammation_score"], vmax=0.1, vmin=-0.1,
           save="_inflammation_fixed.pdf")

       'Atp5f1a', 'Atp5f1b', 'Atp5f1c', 'Atp5f1d', 'Atp5f1e', 'Atp5if1',
       'Atp5me', 'Atp5mf', 'Atp5pb', 'Atp5pd', 'Atp5pf', 'Atp5po', 'Atp6-ps',
       'Ccnb1-ps', 'Chchd2-ps', 'Cox8c', 'Csl', 'Cyp1a2', 'Dhrs2', 'G6pc1',
       'G6pd2', 'Gapdhrt', 'Gapdhrt2', 'Gba1', 'Gyg1', 'Il3', 'Ins1', 'Ins2',
       'Lep', 'Macroh2a1', 'Mc4r', 'Mir451a', 'Mir451b', 'Mt3', 'Mup1',
       'Mup11', 'Mup2', 'Mup3', 'Mup4', 'Mup5', 'Myog', 'Ndufb1', 'Nkx1-1',
       'Norad', 'Oas1b', 'Oas1d', 'Oas1e', 'Oas1f', 'Oxct2a', 'Oxct2b',
       'Pdha2', 'Pgk2', 'Phlda2', 'Pklr', 'Prlh', 'Pth', 'Tafazzin', 'Trex1',
       'Tyrp1', 'Vgf'],
      dtype='object')


       'Ighg2b', 'Ins1', 'Ins2', 'Klk1b1', 'Mrgpra3', 'Nlrp6', 'Npy', 'Npy5r',
       'Orm1', 'Reg3a', 'Reg3b', 'Reg3g', 'Saa1', 'Saa2', 'Serpina1a',
       'Serpina1b', 'Ugt1a1', 'Vnn1', 'Zp3'],
      dtype='object')


## 9. Violins with **computed** significance (fix 2)

Same layout as before — violin + strip, stars over `SED–Ex` and `SED–Fhl2 OE` — but the
p-values are Kruskal–Wallis across the three groups followed by pairwise two-sided
Mann–Whitney U, exactly as the figure legends state.

In [14]:
GROUPS = ["SED", "Ex", r"$\it{Fhl2}$ OE"]
PAIRS  = [("SED", "Ex"), ("SED", r"$\it{Fhl2}$ OE")]

def values_of(ad, feature, from_obs):
    if from_obs:
        return ad.obs[feature].astype(float).values
    return np.asarray(ad.raw[:, feature].X.todense()).ravel()

def violin_with_stats(ad, feature, from_obs=False, shift=0.02, save=None, verbose=True):
    vals = values_of(ad, feature, from_obs)
    grp  = ad.obs["type"].astype(str).values
    data = {g: vals[grp == g] for g in GROUPS}

    H, p_kw = kruskal(*[data[g] for g in GROUPS])
    pvals = []
    for a, b in PAIRS:
        U, p = mannwhitneyu(data[a], data[b], alternative="two-sided")
        pvals.append(p)
    if verbose:
        print(f"### {feature} ###  Kruskal-Wallis H={H:.2f}, p={p_kw:.3e}")
        for (a, b), p in zip(PAIRS, pvals):
            print(f"    {a} vs {b}: p={p:.3e}")
        for a, b in [("Ex", r"$\it{Fhl2}$ OE")]:
            U, p = mannwhitneyu(data[a], data[b], alternative="two-sided")
            print(f"    ({a} vs {b}, not annotated: p={p:.3e})")

    if from_obs:
        sc.pl.violin(ad, keys=feature, groupby="type", order=GROUPS,
                     stripplot=True, jitter=0.4, show=False)
    else:
        sc.pl.violin(ad, keys=feature, groupby="type", order=GROUPS,
                     stripplot=True, jitter=0.4, show=False, use_raw=True)
    ax = plt.gca(); ax.set_ylabel(feature, fontsize=20)

    ann = Annotator(ax, PAIRS, data=ad.obs, x="type", y=vals, order=GROUPS)
    ann.set_pvalues(pvals)          # computed, not assumed
    ann.annotate(line_offset=0.03)
    for child in ax.get_children():
        if isinstance(child, plt.Text) and "*" in child.get_text():
            x, y = child.get_position(); child.set_position((x, y - shift))
    ymin, ymax = ax.get_ylim(); ax.set_ylim(ymin, ymax * 0.96)
    fig = plt.gcf(); fig.set_size_inches(6, 6)
    if save:
        fig.savefig(f"{OUT_FIG}/{save}", bbox_inches="tight")
    plt.show()
    return {"feature": feature, "kruskal_H": H, "kruskal_p": p_kw,
            **{f"p_{a}_vs_{b}": p for (a, b), p in zip(PAIRS, pvals)}}

### Figure 4f — gene scores

In [15]:
res_scores = [violin_with_stats(adata, f, from_obs=True, save=f"F4f_{f}_fixed.pdf")
              for f in ["cardiac_contraction_score", "metabolites_energy_score", "inflammation_score"]]
pd.DataFrame(res_scores)

### cardiac_contraction_score ###  Kruskal-Wallis H=1110.34, p=7.813e-242
    SED vs Ex: p=3.663e-120
    SED vs $\it{Fhl2}$ OE: p=2.007e-242
    (Ex vs $\it{Fhl2}$ OE, not annotated: p=8.273e-15)


p-value annotation legend:
      ns: 5.00e-02 < p <= 1.00e+00
       *: 1.00e-02 < p <= 5.00e-02
      **: 1.00e-03 < p <= 1.00e-02
     ***: 1.00e-04 < p <= 1.00e-03
    ****: p <= 1.00e-04

SED vs. Ex: Custom statistical test, P_val:3.663e-120
SED vs. $\it{Fhl2}$ OE: Custom statistical test, P_val:2.007e-242


### metabolites_energy_score ###  Kruskal-Wallis H=3882.27, p=0.000e+00
    SED vs Ex: p=3.574e-110
    SED vs $\it{Fhl2}$ OE: p=0.000e+00
    (Ex vs $\it{Fhl2}$ OE, not annotated: p=0.000e+00)


p-value annotation legend:
      ns: 5.00e-02 < p <= 1.00e+00
       *: 1.00e-02 < p <= 5.00e-02
      **: 1.00e-03 < p <= 1.00e-02
     ***: 1.00e-04 < p <= 1.00e-03
    ****: p <= 1.00e-04

SED vs. Ex: Custom statistical test, P_val:3.574e-110
SED vs. $\it{Fhl2}$ OE: Custom statistical test, P_val:0.000e+00


### inflammation_score ###  Kruskal-Wallis H=12.22, p=2.217e-03
    SED vs Ex: p=8.168e-01
    SED vs $\it{Fhl2}$ OE: p=7.549e-03
    (Ex vs $\it{Fhl2}$ OE, not annotated: p=1.751e-03)


p-value annotation legend:
      ns: 5.00e-02 < p <= 1.00e+00
       *: 1.00e-02 < p <= 5.00e-02
      **: 1.00e-03 < p <= 1.00e-02
     ***: 1.00e-04 < p <= 1.00e-03
    ****: p <= 1.00e-04

SED vs. Ex: Custom statistical test, P_val:8.168e-01
SED vs. $\it{Fhl2}$ OE: Custom statistical test, P_val:7.549e-03


,feature,kruskal_H,kruskal_p,p_SED_vs_Ex,p_SED_vs_$\it{Fhl2}$ OE
0,cardiac_contraction_score,1110.339725,7.812540e-242,3.663071e-120,2.007204e-242
1,metabolites_energy_score,3882.265796,0.000000e+00,3.574394e-110,0.000000e+00
2,inflammation_score,12.223553,2.216609e-03,8.167721e-01,7.548568e-03


### Figure 4g — individual genes

In [16]:
res_genes = [violin_with_stats(adata, g, from_obs=False, save=f"F4g_{g}_fixed.pdf")
             for g in ["Nppa", "Nppb", "Tlr4", "Mhrt", "Myh6", "Myh7", "Xirp2"]]
pd.DataFrame(res_genes)

### Nppa ###  Kruskal-Wallis H=249.43, p=6.870e-55
    SED vs Ex: p=5.703e-18
    SED vs $\it{Fhl2}$ OE: p=2.468e-55
    (Ex vs $\it{Fhl2}$ OE, not annotated: p=1.563e-10)


p-value annotation legend:
      ns: 5.00e-02 < p <= 1.00e+00
       *: 1.00e-02 < p <= 5.00e-02
      **: 1.00e-03 < p <= 1.00e-02
     ***: 1.00e-04 < p <= 1.00e-03
    ****: p <= 1.00e-04

SED vs. Ex: Custom statistical test, P_val:5.703e-18
SED vs. $\it{Fhl2}$ OE: Custom statistical test, P_val:2.468e-55


### Nppb ###  Kruskal-Wallis H=783.90, p=6.016e-171
    SED vs Ex: p=6.246e-10
    SED vs $\it{Fhl2}$ OE: p=4.849e-149
    (Ex vs $\it{Fhl2}$ OE, not annotated: p=4.581e-104)


p-value annotation legend:
      ns: 5.00e-02 < p <= 1.00e+00
       *: 1.00e-02 < p <= 5.00e-02
      **: 1.00e-03 < p <= 1.00e-02
     ***: 1.00e-04 < p <= 1.00e-03
    ****: p <= 1.00e-04

SED vs. Ex: Custom statistical test, P_val:6.246e-10
SED vs. $\it{Fhl2}$ OE: Custom statistical test, P_val:4.849e-149


### Tlr4 ###  Kruskal-Wallis H=944.76, p=7.062e-206
    SED vs Ex: p=3.122e-171
    SED vs $\it{Fhl2}$ OE: p=3.297e-150
    (Ex vs $\it{Fhl2}$ OE, not annotated: p=7.113e-12)


...

### Myh7 ###  Kruskal-Wallis H=355.04, p=8.024e-78
    SED vs Ex: p=4.074e-03
    SED vs $\it{Fhl2}$ OE: p=6.170e-40
    (Ex vs $\it{Fhl2}$ OE, not annotated: p=3.586e-69)


p-value annotation legend:
      ns: 5.00e-02 < p <= 1.00e+00
       *: 1.00e-02 < p <= 5.00e-02
      **: 1.00e-03 < p <= 1.00e-02
     ***: 1.00e-04 < p <= 1.00e-03
    ****: p <= 1.00e-04

SED vs. Ex: Custom statistical test, P_val:4.074e-03
SED vs. $\it{Fhl2}$ OE: Custom statistical test, P_val:6.170e-40


### Xirp2 ###  Kruskal-Wallis H=1534.10, p=0.000e+00
    SED vs Ex: p=1.080e-109
    SED vs $\it{Fhl2}$ OE: p=0.000e+00
    (Ex vs $\it{Fhl2}$ OE, not annotated: p=5.758e-70)


p-value annotation legend:
      ns: 5.00e-02 < p <= 1.00e+00
       *: 1.00e-02 < p <= 5.00e-02
      **: 1.00e-03 < p <= 1.00e-02
     ***: 1.00e-04 < p <= 1.00e-03
    ****: p <= 1.00e-04

SED vs. Ex: Custom statistical test, P_val:1.080e-109
SED vs. $\it{Fhl2}$ OE: Custom statistical test, P_val:0.000e+00


,feature,kruskal_H,kruskal_p,p_SED_vs_Ex,p_SED_vs_$\it{Fhl2}$ OE
0,Nppa,249.430103,6.869756e-55,5.702592e-18,2.467515e-55
1,Nppb,783.895233,6.016070e-171,6.245856e-10,4.849151e-149
2,Tlr4,944.755661,7.061792e-206,3.121913e-171,3.296810e-150
3,Mhrt,495.982837,1.989278e-108,1.303287e-60,4.087576e-105
4,Myh6,461.423630,6.355238e-101,3.956816e-89,1.579293e-39
5,Myh7,355.038359,8.024167e-78,4.074404e-03,6.170373e-40
6,Xirp2,1534.096802,0.000000e+00,1.079788e-109,0.000000e+00


## 11. Figure 4h — Fhl2 OE vs Ex differential expression and enrichment

In [18]:
sc.tl.rank_genes_groups(adata, groupby="type", groups=[r"$\it{Fhl2}$ OE"],
                        reference="Ex", method="wilcoxon")
rg = adata.uns["rank_genes_groups"]
grp = rg["names"].dtype.names[0]
DEG = pd.DataFrame({"gene": rg["names"][grp],
                    "logfc": rg["logfoldchanges"][grp],
                    "pvals_adj": rg["pvals_adj"][grp]})
DEG = DEG.loc[DEG["pvals_adj"] < 0.05]
DEG_Fhl2 = DEG.loc[DEG["logfc"] > 0.5].copy()
DEG_Ex   = DEG.loc[DEG["logfc"] < -0.5].copy()
DEG_Ex["logfc"] = -DEG_Ex["logfc"]
print("up in Fhl2 OE:", len(DEG_Fhl2), " up in Ex:", len(DEG_Ex))
DEG.to_csv(f"{OUT_AD}/DEG_Fhl2OE_vs_Ex_fixed.csv", index=False)
DEG_Fhl2.head(15)

up in Fhl2 OE: 1826  up in Ex: 558


,gene,logfc,pvals_adj
0,Actn2,1.004209,0.000000e+00
1,Myl3,3.010081,0.000000e+00
2,Tenm3,7.085725,0.000000e+00
3,Mir99ahg,1.029343,0.000000e+00
4,Tpm1,1.582660,0.000000e+00
5,Mast4,1.987493,0.000000e+00
6,Gpcpd1,1.043436,0.000000e+00
7,Nfia,1.108327,0.000000e+00
8,Mylk4,1.956614,1.387014e-305
9,Atp5a1,1.721004,1.546508e-285


In [19]:
import gseapy as gp
geneset_list = ['MSigDB_Hallmark_2020', 'GO_Biological_Process_2023',
                'KEGG_2019_Mouse', 'Reactome_2022']

def enrich(genes, tag):
    if len(genes) < 5:
        print(f"{tag}: too few genes ({len(genes)})"); return None
    e = gp.enrichr(gene_list=list(genes), gene_sets=geneset_list,
                   organism='mouse', outdir=None)
    r = e.results.sort_values('Adjusted P-value').head(20)
    r.to_csv(f"{OUT_AD}/enrichr_{tag}_fixed.csv", index=False)
    fig, ax = plt.subplots(figsize=(7, 6))
    top = r.head(10).iloc[::-1]
    ax.barh(top['Term'].str.slice(0, 55), -np.log10(top['Adjusted P-value']))
    ax.set_xlabel('-log10(adjusted P-value)'); ax.set_title(tag)
    plt.tight_layout(); fig.savefig(f"{OUT_FIG}/F4h_{tag}_fixed.pdf", bbox_inches="tight")
    plt.show()
    return r

res_fhl2 = enrich(DEG_Fhl2["gene"], "up_in_Fhl2OE")
res_ex   = enrich(DEG_Ex["gene"],   "up_in_Ex")

## 12. Save corrected objects

In [20]:
adata_count = adata.copy()
adata_count.X = adata.layers["counts"].copy()
adata_count.write_h5ad(f"{OUT_AD}/adata_Fhl2_count_fixed.h5ad")
adata.write_h5ad(f"{OUT_AD}/adata_Fhl2_analysed_fixed.h5ad")

summary = pd.DataFrame({"n_cells": adata.obs["type"].value_counts()})
summary.to_csv(f"{OUT_AD}/cell_counts_fixed.csv")
print(summary)
print("\nwritten to", OUT_AD)

                n_cells
type                   
$\it{Fhl2}$ OE     4706
Ex                 3456
SED                2772

written to /Users/takahiro/Desktop/project/Reha/snRNAseq_scanpy/Fhl2OE/adata_fixed
